# 04 — Low-Memory Dropout

**Ladder:** Naive materialize mask → fused rand+scale (Python) → **seeded Philox** → Triton `tl.rand` (no mask tensor).

**Study tip.** Prove $\mathbb{E}[Y]=X$ for inverted dropout, then ask what you store instead of an $O(N)$ mask (the seed). Forward/backward must share that seed.

Official: [Low-Memory Dropout](https://triton-lang.org/main/getting-started/tutorials/04-low-memory-dropout.html)

Figures: [`docs/images/`](docs/images/) · guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

# Part I — Mathematics

## 0. Definition and unbiasedness

Fix a keep probability $q=1-p$ with $p\in[0,1)$. Let
$B_i\sim\mathrm{Bernoulli}(q)$ be i.i.d. and define **inverted dropout**

$$
Y_i
=
\begin{cases}
X_i/q, & B_i=1,\\[2pt]
0, & B_i=0,
\end{cases}
\qquad\text{i.e.}\qquad
Y_i=\frac{B_i}{q}X_i .
\tag{0.1}
$$

At evaluation time $p=0$ (so $q=1$) and $Y=X$; no rescaling is needed at
inference, which is the reason for the "inverted" convention.

**Proposition 1 (unbiasedness).** Conditional on $X$,

$$
\mathbb{E}[Y_i\mid X_i]=X_i,
\qquad
\mathbb{E}[Y\mid X]=X.
\tag{0.2}
$$

*Proof.* Since $B_i$ is independent of $X_i$ and
$\mathbb{E}[B_i]=q$,

$$
\mathbb{E}[Y_i\mid X_i]
=
\frac{X_i}{q}\mathbb{E}[B_i]
=
\frac{X_i}{q}\,q
=
X_i.
\qquad\square
$$

**Corollary 2 (unbiased gradient).** For a downstream loss $L$,

$$
\frac{\partial Y_i}{\partial X_i}=\frac{B_i}{q},
\qquad
\mathbb{E}\!\left[\frac{\partial Y_i}{\partial X_i}\right]=1,
\tag{0.3}
$$

so the stochastic gradient is an unbiased estimator of the deterministic
gradient. This is why inverted dropout is safe for training.

### 0.1 Variance and signal-to-noise ratio

**Proposition 3 (variance).** Conditional on $X_i$,

$$
\operatorname{Var}(Y_i\mid X_i)
=
X_i^{2}\,\frac{p}{q}
=
X_i^{2}\,\frac{p}{1-p}.
\tag{0.4}
$$

*Proof.* Since $B_i^{2}=B_i$,

$$
\mathbb{E}[Y_i^{2}\mid X_i]
=\frac{X_i^{2}}{q^{2}}\mathbb{E}[B_i^{2}]
=\frac{X_i^{2}}{q}.
$$

Therefore

$$
\operatorname{Var}
=
\frac{X_i^{2}}{q}-X_i^{2}
=
X_i^{2}\left(\frac{1-q}{q}\right)
=
X_i^{2}\frac{p}{q}.
\qquad\square
$$

**Signal-to-noise ratio.** Define

$$
\mathrm{SNR}_i
=
\frac{|\mathbb{E}[Y_i]|^{2}}{\operatorname{Var}(Y_i)}
=
\frac{q}{p}.
$$

It decreases with $p$: aggressive dropout injects more noise. As
$p\to1$, $q\to0$ and both the scale $1/q$ and the variance blow up — the reason
dropout rates are usually $\le0.5$.

**Corollary 4 (second moment).**

$$
\mathbb{E}[Y_i^{2}\mid X_i]=\frac{X_i^{2}}{q},
\qquad
\mathbb{E}\!\left[\|Y\|_2^{2}\mid X\right]=\frac{\|X\|_2^{2}}{q}.
\tag{0.5}
$$

So inverted dropout preserves the norm in expectation up to the factor $1/q$.

### 0.2 What is random, and what is stored?

The classical implementation draws a full mask

$$
M\in\{0,1\}^{N},
\qquad
M_i=B_i,
\tag{0.6}
$$

and stores it (or recomputes and stores it) for the backward pass. The
low-memory idea is to replace the materialized mask by a **counter-based**
pseudorandom function

$$
U_i=\mathrm{Philox}(\mathrm{seed},\,i)\in[0,1),
\qquad
B_i=\mathbf{1}[U_i>p].
\tag{0.7}
$$

Because $\mathrm{Philox}$ is a pure function of $(\mathrm{seed},i)$, the entire
mask is regenerated on demand from $O(1)$ state. No $O(N)$ buffer is read or
written.

**Assumption (statistical).** For distinct $i$, the outputs $U_i$ are modeled as
i.i.d. $\mathrm{Unif}(0,1)$. Under this model
$\Pr[B_i=1]=\Pr[U_i>p]=1-p=q$, matching §0. In practice Philox is a
well-tested counter-based generator; the model is the standard cryptographic
idealization.

**Reproducibility.** Same seed and same index $\Rightarrow$ same uniform
$\Rightarrow$ same mask. Forward, backward, and activation-checkpoint recompute
therefore agree bit-for-bit, with no saved mask.

### 0.3 Standard vs inverted dropout

| Variant | Train | Inference | Stored state |
|---|---|---|---|
| Standard | $Y=B\odot X$ | $Y=qX$ | mask or seed |
| Inverted | $Y=(B/q)\odot X$ | $Y=X$ | mask or seed |

Both have the same expectation at train time after the inference rescale; the
inverted form moves the rescale into training and is what all modern frameworks
use.

![Whiteboard: materialize mask vs Philox seed in registers](docs/images/benchmark-low-memory-dropout.png)

*Theory board — $O(N)$ mask tensor in DRAM vs seed + `Philox(s,i)` keep/drop in registers.*

### Theory board

![Whiteboard: materialize mask vs Philox seed in registers](docs/images/whiteboard-theory.jpg)

*Theory board — $O(N)$ mask tensor in DRAM vs seed + `Philox(s,i)` keep/drop in registers.*

# Part II — Proof

**Theorem (dropout law).** Let $U_i\sim\mathrm{Unif}(0,1)$ be independent,
$B_i=\mathbf{1}[U_i>p]$, $q=1-p$, and $Y_i=B_iX_i/q$. Then
$Y_i\mid X_i$ has the same law as inverted dropout with keep probability $q$,
and $\mathbb{E}[Y_i\mid X_i]=X_i$.

*Proof.* $\Pr[B_i=1]=\Pr[U_i>p]=1-p=q$, so $B_i\sim\mathrm{Bernoulli}(q)$.
Conditioning on $X_i$ and using independence,
$\mathbb{E}[Y_i]=X_i\mathbb{E}[B_i]/q=X_i$. $\square$

**Corollary (variance).**
$\operatorname{Var}(Y_i\mid X_i)=X_i^2p/q$.

*Proof.* $\mathbb{E}[Y_i^2]=X_i^2\mathbb{E}[B_i]/q^2=X_i^2/q$, so the variance
is $X_i^2(1/q-1)=X_i^2p/q$. $\square$

**Theorem (deterministic regeneration).** If the mask is generated by a pure
function $U_i=\mathrm{Philox}(\mathrm{seed},i)$, then recomputing the function
with the same seed reproduces the same mask exactly.

*Proof.* A pure function has equal outputs for equal inputs; the pair
$(\mathrm{seed},i)$ is identical on every call. $\square$

# Part III — Step-by-step proof

1. **Draw a uniform.** For each index $i$, compute $U_i$ from the seed and $i$.
2. **Threshold.** Set $B_i=\mathbf{1}[U_i>p]$, so the keep probability is
   $q=1-p$.
3. **Scale.** Set $Y_i=B_iX_i/q$; the $1/q$ compensates dropped elements.
4. **Unbiasedness.** Take expectations over $B_i$ to get
   $\mathbb{E}[Y_i]=X_i$.
5. **Variance.** Use $B_i^2=B_i$ to obtain $X_i^2p/q$.
6. **Determinism.** Keep only the seed; regenerate $U_i$ whenever the mask is
   needed.
7. **Memory.** A materialized mask costs $O(N)$ bytes and extra traffic; the
   seeded path costs $O(1)$ state.
8. **Backward.** Apply the same threshold to obtain
   $\partial Y_i/\partial X_i=B_i/q$.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



### Method 01 — Naive (materialized mask)

$$
U_i\sim\mathrm{Unif}(0,1),
\qquad
M_i=\mathbf{1}[U_i>p],
\qquad
Y=\frac{X\odot M}{1-p}.
\tag{1.1}
$$

**Teaching sizes.** Try $N=8$ or $N=1024$, $p=0.5$. The keep rate should track
$1-p$; the mean of many seeded runs of $Y$ should track $X$.

### 1.1 Traffic accounting

Assume $X,Y$ have element size $b$ bytes and the mask is stored as one byte per
element. The naive implementation performs

| Operation | Bytes |
|---|---|
| read $X$ to build the mask | $Nb$ |
| write $M$ | $N$ |
| read $X$ and $M$ to apply | $Nb+N$ |
| write $Y$ | $Nb$ |
| **total** | $3Nb+2N$ |

so

$$
Q_{\mathrm{naive}}=3Nb+2N
=
N(3b+2).
\tag{1.2}
$$

A fused kernel reads $X$ once and writes $Y$ once:

$$
Q_{\mathrm{fused}}=2Nb .
\tag{1.3}
$$

Hence

$$
\frac{Q_{\mathrm{naive}}}{Q_{\mathrm{fused}}}
=
\frac{3b+2}{2b}
=
\frac32+\frac{1}{b}.
\tag{1.4}
$$

For `float32` ($b=4$) the ratio is $14/8=1.75\times$; for `float16` ($b=2$) it
is $8/4=2.0\times$. If the mask were stored in the same dtype as $X$, the naive
total would be $5Nb$, so the ratio would be $5/2=2.5\times$ for every dtype. The bigger cost is often the **allocation**: the mask
is an extra $O(N)$ tensor in the memory planner.

### 1.2 Extra kernel launches

The naive chain is at least two kernels (generate mask, apply mask) plus a
`torch.where`; each launch adds latency $L$ and forces the intermediate to round
trip through HBM. For small tensors the launches dominate:

$$
t_{\mathrm{naive}}\approx 2L+\frac{N(3b+2)}{\mathrm{BW}},
\qquad
t_{\mathrm{fused}}\approx L+\frac{2Nb}{\mathrm{BW}}.
\tag{1.5}
$$

### 1.3 Statistical sanity

For $N$ independent Bernoulli$(q)$ draws, the empirical keep rate
$\hat q=\frac1N\sum_i B_i$ satisfies

$$
\mathbb{E}[\hat q]=q,
\qquad
\operatorname{Var}(\hat q)=\frac{q(1-q)}{N}.
\tag{1.6}
$$

For $N=5\times10^{4}$ and $q=0.75$,
$\operatorname{sd}(\hat q)=\sqrt{0.75\cdot0.25/50000}\approx0.00194$. The test
tolerance $0.02$ is therefore about ten standard deviations — a very safe check,
not a tight one.


In [ ]:
def naive_dropout_pseudo(x, p, seed: int = 0):
    """
    Classic dropout: allocate a full keep-mask, then apply it.

    Memory picture for an n-element tensor:
        - read  x          (n floats)
        - write mask       (n bools)   <-- extra global buffer
        - read  x + mask
        - write y

    The mask itself is often as large as x. That is the cost we want to avoid.
    """
    assert 0.0 <= p < 1.0
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    # Materialize the full Bernoulli mask in memory
    x_cpu = x.detach().cpu()
    mask = torch.rand(x_cpu.shape, generator=g, dtype=x_cpu.dtype) >= p
    y = torch.where(mask, x_cpu / (1.0 - p), torch.zeros_like(x_cpu))
    return y.to(device=x.device, dtype=x.dtype)


### Method 02 — Kernel-based (materialized mask)

This method keeps the classical two-buffer structure but moves both steps to
Triton: one kernel writes the Bernoulli mask to global memory, and a second
kernel reads $X$ and the mask and writes $Y$. It is a genuine kernel
implementation, but it still pays the $O(N)$ mask traffic that Method 03
removes.


In [ ]:
@triton.jit
def dropout_mask_kernel(mask_ptr, n_elements, p, seed, BLOCK_SIZE: tl.constexpr):
    """Method 02, pass 1: materialize the keep-mask in global memory."""
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    uniform = tl.rand(seed, offsets)
    keep = uniform >= p
    tl.store(mask_ptr + offsets, keep.to(tl.int8), mask=mask)


@triton.jit
def dropout_apply_kernel(
    x_ptr, mask_ptr, output_ptr, n_elements, p, BLOCK_SIZE: tl.constexpr
):
    """Method 02, pass 2: read X and the mask, write scaled output."""
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    keep = tl.load(mask_ptr + offsets, mask=mask, other=0)
    y = tl.where(keep != 0, x / (1.0 - p), 0.0)
    tl.store(output_ptr + offsets, y, mask=mask)


def dropout_kernel_based(x, p, seed=0):
    """Two kernels + an O(N) mask buffer (contrast with the fused Method 03)."""
    n = x.numel()
    mask = torch.empty(n, device=x.device, dtype=torch.int8)
    y = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    dropout_mask_kernel[grid](mask, n, p, seed, BLOCK_SIZE=1024)
    dropout_apply_kernel[grid](x, mask, y, n, p, BLOCK_SIZE=1024)
    return y


### Method 03 — Fused kernel-based (Philox)

### 2.1 Intermediate (Python fused)

Per index $i$, one pass — still host RNG, but no persistent mask array:

$$
U_i=\mathrm{rand}(\mathrm{seed},i),
\qquad
Y_i=\mathbf{1}[U_i>p]\,\frac{X_i}{1-p}.
\tag{2.1}
$$

### 2.2 Triton (seeded, in-kernel)

Philox/hash RNG keyed by $(\mathrm{seed},i)$ via `tl.rand(seed, offsets)`:

$$
U_i=\mathrm{tl.rand}(\mathrm{seed},i),
\qquad
Y_i=\mathbf{1}[U_i>p]\,\frac{X_i}{1-p}.
\tag{2.2}
$$

Same distribution as §1 if the $U_i$ are i.i.d. $\mathrm{Unif}(0,1)$. The
uniforms stay in **registers** — there is **no** global $M$.

**Why counter-based RNG?** A stateful generator would need a sequential stream
and could not be evaluated independently per element. Philox maps a counter
(here the offset $i$, optionally combined with the seed) through a bijective
mixing function, so any element can be generated in any order, on any thread,
with no storage:

$$
U_i=\mathrm{Philox}_{k}(\mathrm{seed},i),
\tag{2.3}
$$

where $k$ is the number of mixing rounds. Independence across $i$ is inherited
from the avalanche property of the mixing function.

**Reproducibility.** Same seed $\Rightarrow$ same keep-set on forward, backward,
or activation-checkpoint recompute. Only $O(1)$ RNG state is stored.

### 2.3 Backward pass with the same seed

For a downstream gradient $g=\partial L/\partial Y$, the chain rule through
(0.1) gives

$$
\frac{\partial L}{\partial X_i}
=
g_i\,\frac{B_i}{q}
=
g_i\,\frac{\mathbf{1}[U_i>p]}{q},
\qquad
U_i=\mathrm{Philox}(\mathrm{seed},i).
\tag{2.4}
$$

Thus the backward pass recomputes the mask from the seed instead of loading it.
The expected gradient is unbiased by Corollary 2. In a fused forward/backward
kernel one can even reuse the same registers, so the mask never touches memory
at all.

### 2.4 Cost comparison

| Stage | Function | Mask in DRAM? | Bytes moved |
|-------|----------|---------------|-------------|
| Naive | `naive_dropout_pseudo` | yes | $3Nb+2N$ |
| Fused Python | `fused_dropout_pseudo` | no | $2Nb$ |
| Triton | `dropout_kernel` / `dropout` | no | $2Nb$ |

The Triton kernel is a blocked 1-D map exactly like notebook 01:

$$
\mathrm{offsets}_p=pB+\{0,\dots,B-1\},
\qquad
m=\mathrm{offsets}_p<N,
\tag{2.5}
$$

with one extra register-resident random draw per lane. The arithmetic per
element is one comparison, one select, and one multiply, so the kernel remains
memory-bound.

### 2.5 Numerical detail

The comparison `random >= p` uses the same threshold for every element, so the
realized keep rate is the sample mean (1.6). Values are scaled by $1/q$; with
`float16` this can overflow if $q$ is tiny, which is another reason to keep
$p\le0.5$ in practice. The test `y[nonzero] == x[nonzero] * scale` checks the
exact scaling identity, while the mean/variance checks validate the distribution.


In [ ]:
def fused_dropout_pseudo(x, p, seed: int = 0):
    """
    Same math, but generate the keep decision per element without storing mask.

    We use a Python loop (for teaching): random() and the scale/zero happen in
    one pass — no n-element mask tensor is ever allocated. The Triton kernel
    does the same with tl.rand(seed, offsets).
    """
    assert 0.0 <= p < 1.0
    n = x.numel()
    y = torch.empty_like(x, device="cpu").cpu()
    x_flat = x.detach().cpu().reshape(-1)
    y_flat = y.reshape(-1)

    g = torch.Generator(device="cpu")
    g.manual_seed(seed)

    for i in range(n):
        # Uniform[0, 1) keep decision — never written to a mask buffer
        r = torch.rand((), generator=g).item()
        if r >= p:
            y_flat[i] = x_flat[i] / (1.0 - p)
        else:
            y_flat[i] = 0.0

    return y.to(device=x.device, dtype=x.dtype)


In [ ]:
@triton.jit
def dropout_kernel(
    x_ptr,
    output_ptr,
    n_elements,
    p,
    seed,
    BLOCK_SIZE: tl.constexpr,
):
    # Blocked 1-D grid (same pattern as vector-add)
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    # On-the-fly RNG — no mask tensor in HBM
    random = tl.rand(seed, offsets)
    x_keep = random >= p
    output = tl.where(x_keep, x / (1 - p), 0.0)
    tl.store(output_ptr + offsets, output, mask=mask)


def dropout(x, p, seed=0):
    output = torch.empty_like(x)
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    dropout_kernel[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output


### Pitfalls

- Using `p` vs `1-p` incorrectly in the keep test.
- Materializing a full mask defeats the point of this tutorial.
- Different seeds → different masks; compare distributions, not exact tensors.
- Forgetting to reuse the seed in the backward pass.

### Self-check

1. Show $\mathbb{E}[Y]=X$ and derive $\operatorname{Var}(Y)=X^{2}p/(1-p)$.
2. Where does the boolean keep decision live in the Triton kernel?
3. Why is one integer seed enough to regenerate the mask?
4. What changes if the mask is stored as `float32` instead of `bool`?


In [ ]:
def test_pseudo_statistical() -> None:
    """CPU-runnable: keep rate ≈ (1-p); mean ≈ 1 on a ones vector; seeded fuse is reproducible."""
    torch.manual_seed(0)
    n = 50_000
    x = torch.ones(n)  # ones → expected mean is 1 under inverted dropout
    p = 0.25

    y_naive = naive_dropout_pseudo(x, p, seed=0)
    y_fused = fused_dropout_pseudo(x, p, seed=7)

    keep_naive = (y_naive != 0).float().mean().item()
    keep_fused = (y_fused != 0).float().mean().item()
    assert abs(keep_naive - (1.0 - p)) < 0.02
    assert abs(keep_fused - (1.0 - p)) < 0.02

    # Inverted dropout on ones → kept values are 1/(1-p); overall mean ≈ 1
    assert abs(y_naive.mean().item() - 1.0) < 0.05
    assert abs(y_fused.mean().item() - 1.0) < 0.05

    # p=0 → identity
    assert torch.allclose(naive_dropout_pseudo(x, p=0.0, seed=0), x)
    assert torch.allclose(fused_dropout_pseudo(x, p=0.0, seed=0), x)

    # Seeded reproducibility for fused Python path
    y_fused_2 = fused_dropout_pseudo(x, p, seed=7)
    assert torch.equal(y_fused, y_fused_2)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(4096, device=DEVICE)
    p = 0.25
    seed = 123
    y = dropout(x, p, seed=seed)
    y2 = dropout(x, p, seed=seed)
    # Same seed → bitwise-identical output (tl.rand is deterministic given seed+offset)
    assert torch.equal(y, y2)

    # Kept values must be x / (1-p)
    scale = 1.0 / (1.0 - p)
    nonzero = y != 0
    assert torch.allclose(y[nonzero], x[nonzero] * scale, atol=1e-5)

    # Statistical sanity on a larger buffer of ones
    ones = torch.ones(100_000, device=DEVICE)
    y_stat = dropout(ones, p, seed=seed)
    keep_rate = (y_stat != 0).float().mean().item()
    assert abs(keep_rate - (1.0 - p)) < 0.02
    assert abs(y_stat.mean().item() - 1.0) < 0.05
    print("deterministic dropout OK", y.count_nonzero().item())


In [ ]:
def test_kernel_methods_dropout() -> None:
    """Method 02 (materialized mask) and Method 03 (fused Philox) agree."""
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(10000, device=DEVICE)
    for p in (0.0, 0.25, 0.5):
        y_kernel = dropout_kernel_based(x, p, seed=7)
        y_fused = dropout(x, p, seed=7)
        # Same seed + same offsets => identical keep decisions.
        assert torch.equal(y_kernel, y_fused), p
    print("dropout: methods 02 and 03 agree")
